In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.3 Vocabulary design

Vocabulary size is a budget. A bigger vocab learns more merges, so it encodes
text in fewer tokens, but every token costs an embedding row and a logit. The
engineering job is choosing the trade.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
sample = corpus[:20000]
small = BPETokenizer().train(sample, 300)
big = BPETokenizer().train(sample, 600)
probe = corpus[20000:21000]
print("vocab 300 ->", len(small.encode(probe)), "tokens")
print("vocab 600 ->", len(big.encode(probe)), "tokens")

Greedy BPE is nested: the 600-token vocab contains every merge the 300-token
vocab learned, plus more, so it can only encode in fewer-or-equal tokens.

In [ ]:
assert len(big.encode(probe)) <= len(small.encode(probe))